In [1]:
!pip install -q ifcopenshell pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 MB 18.5 MB/s eta 0:00:00


Saving IFC Schependomlaan incl planningsdata.ifc to IFC Schependomlaan incl planningsdata.ifc
Uploaded: IFC Schependomlaan incl planningsdata.ifc


In [4]:
import pandas as pd
import ifcopenshell
import ifcopenshell.util.element

model = ifcopenshell.open(ifc_filename)
elements = model.by_type("IfcElement")

print("IFC schema:", model.schema)
print("Number of elements:", len(elements))

IFC schema: IFC2X3
Number of elements: 3505


In [9]:
records = []

for element in elements:
    storey = ifcopenshell.util.element.get_container(
        element,
        ifc_class="IfcBuildingStorey"
    )

    element_type = ifcopenshell.util.element.get_type(
        element
    )

    records.append({
        "GUID": element.GlobalId,
        "IfcClass": element.is_a(),
        "ElementName": getattr(element, "Name", None),
        "Description": getattr(element, "Description", None),
        "ObjectType": getattr(element, "ObjectType", None),
        "Tag": getattr(element, "Tag", None),
        "TypeName": (
            element_type.Name
            if element_type
            else None
        ),
        "BuildingStorey": (
            storey.Name
            if storey
            else None
        ),
        "HasGeometry": (
            getattr(element, "Representation", None)
            is not None
        )
    })

asplanned_bim = pd.DataFrame(records)

display(asplanned_bim.head())

,GUID,IfcClass,ElementName,Description,ObjectType,Tag,TypeName,BuildingStorey,HasGeometry
0,3K0ZfH$5P9$AY9tdZGvNY_,IfcBeam,staal halfspant_(#616505),None,None,DCED1AAC-D669-4C26-B724-FA92628934FF,None,Storey-1,True
1,2_RPGurtHDWvGWEyfSnyy2,IfcBeam,HEA180_(#698107),None,None,E7C8FDBC-37F7-4ADD-8C9D-7BDE872CC079,None,Storey-1,True
2,1qWfbRwbT55R6UZM54lizh,IfcBeam,stripstaal 80/8_(#738386),None,None,89498683-A6D9-4398-BD59-8090C97B0A66,None,Storey-1,True
3,0AjFifcjjCxe8St8Gse3kP,IfcBeam,HEA180_(#696032),None,None,36A62D19-20D3-4A77-BBC1-E041D20418A7,None,Storey-1,True
4,1dvWk_zfz56uvSpgmdhxb0,IfcBeam,stripstaal 80/8_(#737434),None,None,1EEED7EA-00C7-4CD0-A808-E10748E93B71,None,Storey-1,True


In [10]:
print("Total elements:", len(asplanned_bim))
print("Unique GUIDs:", asplanned_bim["GUID"].nunique())
print("Missing GUIDs:", asplanned_bim["GUID"].isna().sum())
print(
    "Duplicated GUIDs:",
    asplanned_bim["GUID"].duplicated().sum()
)

print(
    "Elements with storey information:",
    asplanned_bim["BuildingStorey"].notna().sum()
)

Total elements: 3505
Unique GUIDs: 3505
Missing GUIDs: 0
Duplicated GUIDs: 0
Elements with storey information: 3505


In [11]:
asplanned_bim.to_csv(
    "asplanned_ifc_elements_extracted.csv",
    index=False
)

from google.colab import files

files.download(
    "asplanned_ifc_elements_extracted.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>